# Stress Prediction — 7번째 파생피처 후보 테스트

우리 v34: 원본 9개(age, height, weight, systolic_blood_pressure, diastolic_blood_pressure,
cholesterol, glucose, bone_density, mean_working) + 파생 6개(bmi, pulse_pressure,
mean_arterial_pressure, blood_pressure_ratio, cholesterol_glucose_ratio, missing_count) = 15개

2등 팀: 원본 9개 + 파생 **7개** = 16개. 우리보다 파생피처가 1개 부족합니다.

정확히 어떤 피처인지는 모르니, 임상/생리학적으로 근거 있는 후보 5개를 **하나씩** 추가해서
어떤 게 CV MAE를 낮추는지 확인합니다 (한꺼번에 넣지 않음 — 예전에 그렇게 해서 신호가
묻혔던 경험이 있어서, 이번엔 개별 효과를 정확히 봅니다).

**후보 목록**:
1. `bone_density_age_residual` — 나이 대비 골밀도가 기대치보다 높은지/낮은지 (잔차)
2. `working_per_age` — 나이 대비 활동/근무 강도 (mean_working / age)
3. `bp_product` — 수축기x이완기 혈압 곱 (심박출량 부담의 대리 지표)
4. `age_bmi_interaction` — 나이 x BMI (고령+고BMI 복합 위험)
5. `metabolic_risk_score` — 혈당+콜레스테롤+BMI 세 지표의 z-score 합 (연속형, allostatic_load와 달리 이진화 안 함)


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 800  # 탐색용. 최종 확정 후 1200으로 재확인
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_base_features(frame: pd.DataFrame) -> pd.DataFrame:
    """v34 원본 파생 6개 (모든 실험의 공통 기반)."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_candidate_7th(train_frame, other_frame, candidate_name, train_stats=None):
    """candidate_name에 해당하는 7번째 파생피처를 train/other 양쪽에 동일하게 추가.
    train_stats가 필요한 경우(잔차, z-score) train에서만 계산한 통계를 넘겨받아 사용 (leakage 방지)."""
    train_result = train_frame.copy()
    other_result = other_frame.copy()

    if candidate_name == "bone_density_age_residual":
        # train의 age~bone_density 선형관계로 예측치를 구하고, 잔차(실제-예측)를 피처로
        slope, intercept = np.polyfit(train_frame["age"], train_frame["bone_density"], 1)
        train_result["bone_density_age_residual"] = train_frame["bone_density"] - (slope * train_frame["age"] + intercept)
        other_result["bone_density_age_residual"] = other_frame["bone_density"] - (slope * other_frame["age"] + intercept)

    elif candidate_name == "working_per_age":
        train_result["working_per_age"] = train_frame["mean_working"] / (train_frame["age"] + 1e-6)
        other_result["working_per_age"] = other_frame["mean_working"] / (other_frame["age"] + 1e-6)

    elif candidate_name == "bp_product":
        train_result["bp_product"] = train_frame["systolic_blood_pressure"] * train_frame["diastolic_blood_pressure"]
        other_result["bp_product"] = other_frame["systolic_blood_pressure"] * other_frame["diastolic_blood_pressure"]

    elif candidate_name == "age_bmi_interaction":
        train_result["age_bmi_interaction"] = train_frame["age"] * train_frame["bmi"]
        other_result["age_bmi_interaction"] = other_frame["age"] * other_frame["bmi"]

    elif candidate_name == "metabolic_risk_score":
        cols = ["glucose", "cholesterol", "bmi"]
        z_sum_train = np.zeros(len(train_frame))
        z_sum_other = np.zeros(len(other_frame))
        for col in cols:
            mean, std = train_frame[col].mean(), train_frame[col].std()
            z_sum_train += (train_frame[col] - mean) / (std + 1e-9)
            z_sum_other += (other_frame[col] - mean) / (std + 1e-9)
        train_result["metabolic_risk_score"] = z_sum_train
        other_result["metabolic_risk_score"] = z_sum_other

    else:
        raise ValueError(f"unknown candidate: {candidate_name}")

    return train_result, other_result


In [2]:

def add_feature_copies(frame: pd.DataFrame, feature_copy_counts) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in feature_copy_counts.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target, feature_copy_counts) -> np.ndarray:
    weighted_train = add_feature_copies(train_features, feature_copy_counts)
    weighted_val = add_feature_copies(val_features, feature_copy_counts)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## 5-Fold CV — baseline(파생6개) vs 후보 5개 각각 추가(파생7개)

In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_base_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

# pair는 후보와 무관하므로 fold당 한 번만 계산
pair_cache = []
for train_idx, val_idx in cv.split(raw_X):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    pred_pair = pair_prediction(train_feat, val_feat, y_train)
    pair_cache.append((train_idx, val_idx, pred_pair, y_train))
print("pair 캐시 완료")


def cv_mae_baseline():
    oof = np.zeros(len(y))
    for train_idx, val_idx, pred_pair, y_train in pair_cache:
        train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
        val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
        pred_tree = tree_prediction(train_feat, val_feat, y_train, BASE_FEATURE_COPY_COUNTS)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    return mean_absolute_error(y, oof)


def cv_mae_with_candidate(candidate_name):
    oof = np.zeros(len(y))
    feature_copy_counts = {**BASE_FEATURE_COPY_COUNTS, candidate_name: 2}
    for train_idx, val_idx, pred_pair, y_train in pair_cache:
        train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
        val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
        train_feat_c, val_feat_c = add_candidate_7th(train_feat, val_feat, candidate_name)
        pred_tree = tree_prediction(train_feat_c, val_feat_c, y_train, feature_copy_counts)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    return mean_absolute_error(y, oof)


baseline_mae = cv_mae_baseline()
print(f"### baseline(파생 6개) CV MAE: {baseline_mae:.6f}")
print()

candidates = [
    "bone_density_age_residual", "working_per_age", "bp_product",
    "age_bmi_interaction", "metabolic_risk_score",
]
rows = []
for cand in candidates:
    mae = cv_mae_with_candidate(cand)
    rows.append({"candidate_7th_feature": cand, "cv_mae": mae, "delta_vs_baseline": mae - baseline_mae})
    print(f"{cand}: {mae:.6f} (delta {mae - baseline_mae:+.6f})")

result_df = pd.DataFrame(rows).sort_values("cv_mae")
print()
display(result_df)


pair 캐시 완료
### baseline(파생 6개) CV MAE: 0.147993

bone_density_age_residual: 0.147950 (delta -0.000043)
working_per_age: 0.149613 (delta +0.001620)
bp_product: 0.148707 (delta +0.000713)
age_bmi_interaction: 0.148130 (delta +0.000137)
metabolic_risk_score: 0.147997 (delta +0.000003)



,candidate_7th_feature,cv_mae,delta_vs_baseline
0,bone_density_age_residual,0.147950,-0.000043
4,metabolic_risk_score,0.147997,0.000003
3,age_bmi_interaction,0.148130,0.000137
2,bp_product,0.148707,0.000713
1,working_per_age,0.149613,0.001620


## 최종 제출 파일 생성

가장 낮은 `cv_mae`를 준 후보를 `BEST_CANDIDATE`에 넣고 실행하세요. delta가 마이너스로 확실히 나오지 않으면
(오늘 다른 실험들처럼 노이즈 수준이면) 실행하지 말고 v34를 유지하세요.


In [4]:

BEST_CANDIDATE = "bone_density_age_residual"  # 결과 1위로 수정
FINAL_N_ESTIMATORS = 1200

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_base_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

N_ESTIMATORS = FINAL_N_ESTIMATORS
final_feature_copy_counts = {**BASE_FEATURE_COPY_COUNTS, BEST_CANDIDATE: 2}
final_train_feat, final_test_feat = add_candidate_7th(raw_X, raw_X_test, BEST_CANDIDATE)

final_pair = pair_prediction(raw_X, raw_X_test, y)
final_tree = tree_prediction(final_train_feat, final_test_feat, y, final_feature_copy_counts)
final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_7thfeature_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_7thfeature_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.49
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.59
9,TEST_0009,0.88
